# Cython: от Python до C — каждый факт доказан кодом

**Принцип:** каждое утверждение сопровождается ячейкой, которая его доказывает.

---
## Содержание
1. PyFloatObject = 24 байта — **доказательство кодом**
2. Boxing = 6 шагов — **доказательство через `dis`**
3. Cython untyped: boxing остаётся — **доказательство через `dis` + замер**
4. Cython typed: C-переменные на стеке — **замер + сгенерированный C**
5. Typed memoryviews — прямой указатель, важность `ascontiguousarray`
6. Игра в кости — живой бенчмарк всех 3 уровней (понятный эксперимент)
7. Free-threading и `nogil` — параллелизм без GIL
8. Память: list vs NumPy — **измерение tracemalloc**
9. Ограничения Cython


In [ ]:
import sys, subprocess
try:
    import Cython
    print(f'Cython {Cython.__version__} OK')
except ImportError:
    print('Cython не установлен: pip install cython')

# Проверяем C-компилятор (gcc или MSVC)
compiler_found = False
try:
    r = subprocess.run(['gcc', '--version'], capture_output=True, text=True)
    if r.returncode == 0:
        print('gcc OK:', r.stdout.split('\n')[0])
        compiler_found = True
except FileNotFoundError:
    pass

if not compiler_found:
    try:
        r2 = subprocess.run(['cl'], capture_output=True, text=True, shell=True)
        if 'Microsoft' in r2.stderr or r2.returncode == 0:
            print('MSVC (cl.exe) OK')
            compiler_found = True
    except Exception:
        pass

if not compiler_found:
    print('C-компилятор не найден в PATH.')
    print('%%cython ячейки используют компилятор из окружения Jupyter — обычно работает.')
    print('Если нет — установите MSVC Build Tools или MinGW.')


In [ ]:
# Загружаем Cython magic — ОБЯЗАТЕЛЬНО перед %%cython ячейками
%load_ext Cython
print('Cython magic загружен. Теперь %%cython ячейки будут работать.')


---
## 1  PyFloatObject = 24 байта — доказательство

Утверждение: каждый Python `float` — heap-объект размером **24 байта**:
```c
typedef struct {
    Py_ssize_t   ob_refcnt;   // 8 байт — счётчик ссылок
    PyTypeObject *ob_type;    // 8 байт — указатель на тип
    double        ob_fval;    // 8 байт — само значение
} PyFloatObject;
```
Проверим это прямо в Python:


In [ ]:
import sys, ctypes, numpy as np

x = 1.0
print(f'sys.getsizeof(1.0)  = {sys.getsizeof(x)} байт  <- PyFloatObject')
print(f'sys.getsizeof(1)    = {sys.getsizeof(1)} байт  <- PyLongObject')
print(f'ctypes.c_double     = {ctypes.sizeof(ctypes.c_double)} байт  <- C double')
print()
print(f'Overhead: {sys.getsizeof(x)} / {ctypes.sizeof(ctypes.c_double)} = '
      f'{sys.getsizeof(x)/ctypes.sizeof(ctypes.c_double):.1f}x больше памяти')
print()

# Список из 1000 float vs numpy array
n = 1000
py_list = [float(i) for i in range(n)]
np_arr  = np.arange(n, dtype=np.float64)

list_overhead  = sys.getsizeof(py_list)
list_elements  = sum(sys.getsizeof(v) for v in py_list)
list_total     = list_overhead + list_elements

print(f'Список из {n} float:')
print(f'  sys.getsizeof(list)    = {list_overhead} байт  (только указатели)')
print(f'  сумма getsizeof(elem)  = {list_elements} байт  (PyFloatObject x {n})')
print(f'  итого                  = {list_total} байт = {list_total/1024:.1f} KB')
print()
print(f'NumPy array из {n} float64:')
print(f'  np_arr.nbytes          = {np_arr.nbytes} байт = {np_arr.nbytes/1024:.1f} KB')
print(f'  байт на элемент        = {np_arr.nbytes/n:.0f}')
print()
print(f'Экономия памяти NumPy vs list: {list_total/np_arr.nbytes:.1f}x')


---
## 2  Boxing = 6 шагов — доказательство через `dis`

Утверждение: `a + b` в Python — **6 шагов** вместо 1 инструкции CPU.

Модуль `dis` показывает байткод CPython:


In [ ]:
import dis

def add_floats(a, b):
    return a + b

print('=== Байткод Python: a + b ===')
dis.dis(add_floats)
print()
print('BINARY_OP (+) разворачивается в:')
print('  1. Py_TYPE(a)->tp_as_number->nb_add(a, b)  <- vtable lookup')
print('  2. ((PyFloatObject*)a)->ob_fval             <- unbox a')
print('  3. ((PyFloatObject*)b)->ob_fval             <- unbox b')
print('  4. double + double                          <- 1 инструкция CPU')
print('  5. PyFloat_FromDouble(result)               <- malloc + init')
print('  6. Py_INCREF / Py_DECREF                   <- атомарные GC')


In [ ]:
# Докажем счётчик ссылок — первые 8 байт PyFloatObject
import ctypes

x = 3.14
print(f'id(x) = {id(x)}  <- адрес PyFloatObject на куче')

refcnt = ctypes.c_long.from_address(id(x)).value
print(f'ob_refcnt = {refcnt}  <- CPython отслеживает каждую ссылку')

y = x  # создаём ещё одну ссылку
refcnt2 = ctypes.c_long.from_address(id(x)).value
print(f'ob_refcnt после y = x: {refcnt2}  <- увеличился на 1')

del y
refcnt3 = ctypes.c_long.from_address(id(x)).value
print(f'ob_refcnt после del y: {refcnt3}  <- уменьшился обратно')
print()
print('Вывод: каждое присваивание = INCREF, каждый del = DECREF.')
print('В цикле из 1M итераций это 1M INCREF + 1M DECREF = 2M атомарных операций.')


---
## 3  Cython untyped: boxing ОСТАЁТСЯ — доказательство

Утверждение: Cython без `cdef` компилирует в C, но переменные **всё ещё PyObject***.

Докажем через `dis` и замер времени:


In [ ]:
%%cython --quiet
# Cython untyped: нет cdef -> переменные PyObject*

def cy_add_untyped(a, b):
    return a + b

def cy_loop_untyped(n):
    s = 0.0          # PyObject* на куче, 24 байта
    for i in range(n):
        s += i * i   # PyNumber_Multiply + PyNumber_Add
    return s


In [ ]:
import dis

print('=== Байткод cy_add_untyped (Cython без типов) ===')
dis.dis(cy_add_untyped)
print()
print('Видим BINARY_OP — тот же байткод что и в чистом Python!')
print('Cython скомпилировал в C, но C-код всё равно вызывает PyNumber_Add.')
print(f'type(cy_add_untyped) = {type(cy_add_untyped)}')


In [ ]:
import time

def py_loop(n):
    s = 0.0
    for i in range(n):
        s += i * i
    return s

def bench(fn, *args, repeats=5):
    best = float('inf')
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best * 1000

N = 500_000
t_py   = bench(py_loop, N)
t_cy_u = bench(cy_loop_untyped, N)

print(f'N = {N:,}')
print(f'Pure Python    : {t_py:.2f} ms  (1.0x)')
print(f'Cython untyped : {t_cy_u:.2f} ms  ({t_py/t_cy_u:.1f}x)')
print()
print('Вывод: ~1.3-2x — убирается только dispatch loop интерпретатора.')
print('Boxing PyObject* остаётся -> нет большого прироста.')


---
## 4  Cython typed: C-переменные на стеке — доказательство

Утверждение: `cdef double x` создаёт переменную **на стеке**, не на куче.

Докажем через замер времени и просмотр сгенерированного C-кода:


In [ ]:
%%cython --quiet
# Cython typed: cdef -> C-переменные на стеке

def cy_loop_typed(int n):
    cdef long long s = 0   # стек, 8 байт, не PyObject
    cdef int i             # стек, 4 байта
    for i in range(n):     # C for-loop: for(i=0; i<n; i++)
        s += i * i         # одна инструкция IMUL + ADD
    return s               # boxing только при return (один раз)


In [ ]:
import dis

print('=== Байткод cy_loop_typed (Cython typed) ===')
dis.dis(cy_loop_typed)
print()
print('Нет BINARY_OP! Весь цикл выполняется в C без байткода.')
print('Байткод нужен только для вызова функции из Python.')


In [ ]:
t_cy_t = bench(cy_loop_typed, N)

print(f'N = {N:,}')
print(f'Pure Python    : {t_py:.2f} ms  (1.0x)')
print(f'Cython untyped : {t_cy_u:.2f} ms  ({t_py/t_cy_u:.1f}x)')
print(f'Cython typed   : {t_cy_t:.3f} ms  ({t_py/t_cy_t:.0f}x)')
print()
print('Вывод: typed в ~100x быстрее Python и в ~50x быстрее untyped.')
print('cdef убирает: boxing, malloc, GC, vtable dispatch.')


In [ ]:
# Смотрим сгенерированный C-код — доказательство что переменные на стеке
import os

root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
c_typed   = os.path.join(root, 'src', 'cy_typed',   'matrix_ops_typed.c')
c_untyped = os.path.join(root, 'src', 'cy_untyped', 'matrix_ops_cy.c')

def show_c_vars(path, label, keywords, max_lines=10):
    if not os.path.exists(path):
        print(f'{label}: файл не найден ({path})')
        return
    with open(path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    print(f'=== {label} ===')
    shown = 0
    for line in lines:
        if any(k in line for k in keywords) and shown < max_lines:
            print(f'  {line.rstrip()}')
            shown += 1
    print()

# Typed: ищем объявления C-переменных (double, int на стеке)
show_c_vars(c_typed, 'TYPED — C-переменные на стеке',
            ['double __pyx_v_x', 'double __pyx_v_y', 'int __pyx_v_inside',
             'int __pyx_v_i', 'double __pyx_v_s', 'long long __pyx_v_s'])

# Untyped: ищем PyObject* объявления
show_c_vars(c_untyped, 'UNTYPED — PyObject* на куче',
            ['PyObject *__pyx_v_s', 'PyObject *__pyx_v_x',
             'PyObject *__pyx_v_y', 'PyObject *__pyx_v_inside'])

print('Typed:   double __pyx_v_x  <- стек, 8 байт')
print('Untyped: PyObject *__pyx_v_x <- куча, 24 байта + malloc')


---
## 5  Typed Memoryviews — прямой указатель на буфер NumPy

`double[::1] a` — структура: указатель + шаг + размер.
Доступ `a[i]` = `*(ptr + i * stride)` — одна инструкция.

**Важно:** `np.ascontiguousarray(arr, dtype=np.float64)` — иначе `BufferError`.


In [ ]:
%%cython --quiet
import numpy as np
cimport numpy as cnp

def dot_slow(a, b):
    s = 0.0
    for i in range(len(a)):
        s += a[i] * b[i]   # PyObject_GetItem -> boxing
    return s

def dot_fast(double[::1] a, double[::1] b):
    cdef int n = a.shape[0]
    cdef int i
    cdef double s = 0.0
    for i in range(n):
        s += a[i] * b[i]   # *(ptr + i*8) — одна инструкция
    return s


In [ ]:
import numpy as np

N = 100_000
a = np.ascontiguousarray(np.random.rand(N), dtype=np.float64)
b = np.ascontiguousarray(np.random.rand(N), dtype=np.float64)

t_slow = bench(dot_slow, a, b)
t_fast = bench(dot_fast, a, b)
t_np   = bench(np.dot, a, b)

print(f'dot(N={N:,}):')
print(f'  Cython без memoryview : {t_slow:.2f} ms  (1.0x)')
print(f'  Cython typed memview  : {t_fast:.3f} ms  ({t_slow/t_fast:.0f}x)')
print(f'  NumPy np.dot          : {t_np:.3f} ms  ({t_slow/t_np:.0f}x)')
print()
print('NumPy быстрее за счёт SIMD/AVX векторизации.')
print()

# Докажем важность ascontiguousarray
print('=== Важность np.ascontiguousarray ===')
a_slice = a[::2]  # не C-contiguous!
print(f'a[::2].flags[C_CONTIGUOUS] = {a_slice.flags["C_CONTIGUOUS"]}')
print(f'a.flags[C_CONTIGUOUS]      = {a.flags["C_CONTIGUOUS"]}')
try:
    dot_fast(a_slice, a_slice)
    print('Сработало')
except Exception as e:
    print(f'Ошибка: {type(e).__name__}: {e}')
    print('Решение: np.ascontiguousarray(a_slice, dtype=np.float64)')


---
## 6  Игра в кости — живой бенчмарк всех 3 уровней

### Правила (понятны любому):
- Бросаем **2 кубика** за раунд
- **Выигрываем**, если сумма ≥ 7
- **Проигрываем**, если сумма ≤ 6

Запускаем N раундов и считаем вероятность выигрыша.

**Истинная вероятность = 21/36 ≈ 0.5833** (21 из 36 комбинаций дают сумму ≥ 7).

Это **идеальный бенчмарк boxing**: только целочисленные скалярные операции.
В Python каждые `die1`, `die2`, `wins` — `PyObject*` на куче.


In [ ]:
# Докажем теорию: перебираем все 36 комбинаций
wins_theory = sum(1 for d1 in range(1,7) for d2 in range(1,7) if d1+d2 >= 7)
print(f'Теория: {wins_theory}/36 = {wins_theory/36:.4f}')
print('Комбинации с суммой >= 7:')
combos = [(d1, d2, d1+d2) for d1 in range(1,7) for d2 in range(1,7) if d1+d2 >= 7]
print(f'  {combos[:6]} ...')
print(f'  Итого: {len(combos)} комбинаций из 36')


In [ ]:
import random

# Уровень 0: Pure Python
# die1, die2, wins — PyObject* на куче, каждый randint = вызов Python
def py_dice_game(n_rounds):
    wins = 0
    for _ in range(n_rounds):
        die1 = random.randint(1, 6)   # PyLongObject, 28 байт
        die2 = random.randint(1, 6)   # PyLongObject, 28 байт
        if die1 + die2 >= 7:          # boxing: unbox -> add -> box -> compare
            wins += 1
    return wins / n_rounds

MC_N = 500_000
prob_py = py_dice_game(MC_N)
t_mc_py = bench(py_dice_game, MC_N)
print(f'Python (N={MC_N:,}): win_prob={prob_py:.4f}  time={t_mc_py:.1f} ms')
print(f'Теория:              win_prob=0.5833')


In [ ]:
%%cython --quiet
# Уровень 1: Cython untyped — die1, die2, wins всё ещё PyObject*
# random.randint() — Python-вызов, boxing остаётся

def cy_dice_untyped(int n_rounds):
    import random
    wins = 0
    for _ in range(n_rounds):
        die1 = random.randint(1, 6)   # PyObject* — boxing остаётся
        die2 = random.randint(1, 6)   # PyObject* — boxing остаётся
        if die1 + die2 >= 7:
            wins += 1
    return wins / n_rounds


In [ ]:
%%cython --quiet
# Уровень 2: Cython typed — всё на стеке, нет PyObject*
# rand() % 6 + 1 — C-уровень, нет Python random module
from libc.stdlib cimport rand, srand

def cy_dice_typed(int n_rounds):
    cdef int i
    cdef int wins = 0
    cdef int die1, die2          # C int на стеке — нет PyObject*
    srand(42)
    with nogil:                  # отпускаем GIL — чистый C
        for i in range(n_rounds):
            die1 = rand() % 6 + 1    # C int, нет boxing
            die2 = rand() % 6 + 1    # C int, нет boxing
            if die1 + die2 >= 7:     # C сравнение, нет PyObject*
                wins += 1
    return <double>wins / n_rounds


In [ ]:
t_mc_u = bench(cy_dice_untyped, MC_N)
t_mc_t = bench(cy_dice_typed, MC_N)

prob_u = cy_dice_untyped(MC_N)
prob_t = cy_dice_typed(MC_N)

print(f'Игра в кости (N={MC_N:,} раундов):')
print(f'  Pure Python    : {t_mc_py:.1f} ms  win_prob={prob_py:.4f}  (1.0x)')
print(f'  Cython untyped : {t_mc_u:.1f} ms  win_prob={prob_u:.4f}  ({t_mc_py/t_mc_u:.1f}x)')
print(f'  Cython typed   : {t_mc_t:.2f} ms  win_prob={prob_t:.4f}  ({t_mc_py/t_mc_t:.0f}x)')
print()
print('Untyped: ~1x — boxing остаётся (die1, die2 — PyObject*, random.randint = Python-вызов)')
print('Typed:   ~50-80x — cdef int die1, die2 + rand()%6+1 + nogil')
print(f'Все три дают правильный ответ: ~0.5833 (теория = 21/36)')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# --- Левый график: распределение сумм двух кубиков ---
rng = np.random.default_rng(42)
n_show = 50_000
d1 = rng.integers(1, 7, n_show)
d2 = rng.integers(1, 7, n_show)
sums = d1 + d2

counts = [np.sum(sums == s) for s in range(2, 13)]
colors_bar = ['#e74c3c' if s < 7 else '#2ecc71' for s in range(2, 13)]
ax1.bar(range(2, 13), counts, color=colors_bar, edgecolor='black', alpha=0.85)
ax1.axvline(6.5, color='black', linestyle='--', linewidth=2, label='граница (сумма=7)')
ax1.set_xlabel('Сумма двух кубиков')
ax1.set_ylabel('Количество выпадений')
ax1.set_title('Распределение сумм (красный=проигрыш, зелёный=выигрыш)', fontweight='bold')
ax1.set_xticks(range(2, 13))
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
win_rate = np.sum(sums >= 7) / n_show
ax1.text(9, max(counts)*0.85, f'Выигрыш: {win_rate:.1%}\n(теория: 58.3%)',
         fontsize=11, color='#27ae60', fontweight='bold')

# --- Правый график: speedup ---
labels   = ['Pure Python', 'Cython untyped', 'Cython typed']
times    = [t_mc_py, t_mc_u, t_mc_t]
colors   = ['#e74c3c', '#e67e22', '#2ecc71']
speedups = [t_mc_py / t for t in times]

bars = ax2.bar(labels, speedups, color=colors, edgecolor='black', alpha=0.85)
ax2.set_ylabel('Ускорение vs Pure Python')
ax2.set_title('Speedup: игра в кости', fontweight='bold')
ax2.axhline(1, color='red', linestyle='--', alpha=0.5)
ax2.grid(axis='y', alpha=0.3)
for bar, s in zip(bars, speedups):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{s:.1f}x', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 7  Free-threading и `nogil` — параллелизм без GIL

### Что такое GIL?

**GIL (Global Interpreter Lock)** — мьютекс в CPython, который гарантирует,
что только **один поток** выполняет Python-байткод в любой момент времени.

```
Thread 1: [Python] --GIL--> [Python] --GIL--> ...
Thread 2:          waiting           waiting   ...
```

Из-за GIL Python-потоки **не дают параллельного ускорения** на CPU-bound задачах.

### `with nogil:` в Cython

Когда Cython-код работает только с C-типами (нет PyObject*), он может **отпустить GIL**:

```cython
with nogil:
    # Здесь нет Python-объектов
    # Другие потоки могут выполняться параллельно
    for i in range(n):
        result += heavy_c_computation(i)
# GIL возвращается автоматически
```

### Python 3.13+ Free-threading (PEP 703)

Python 3.13 добавил **экспериментальный режим без GIL** (`python3.13t`).
Cython `with nogil:` — это **предшественник** этой идеи, работающий уже сейчас.

| | CPython 3.12 | CPython 3.13t (no-GIL) | Cython `with nogil:` |
|---|---|---|---|
| Параллельные потоки | нет (GIL) | да | да (в nogil блоке) |
| Требует изменений кода | нет | нет | да (cdef + nogil) |
| Стабильность | стабильно | экспериментально | стабильно |
| Overhead | нет | ~10-40% | нет |


In [ ]:
# Доказательство 1: GIL блокирует параллелизм в Python
import threading, time

def cpu_bound_python(n):
    s = 0.0
    for i in range(n):
        s += i * i
    return s

N = 2_000_000

# Однопоточно
t0 = time.perf_counter()
cpu_bound_python(N)
cpu_bound_python(N)
t_single = time.perf_counter() - t0

# Двухпоточно (должно быть ~= однопоточному из-за GIL)
t0 = time.perf_counter()
t1 = threading.Thread(target=cpu_bound_python, args=(N,))
t2 = threading.Thread(target=cpu_bound_python, args=(N,))
t1.start(); t2.start()
t1.join();  t2.join()
t_double = time.perf_counter() - t0

print(f'CPU-bound Python (N={N:,} x2):')
print(f'  Однопоточно  : {t_single*1000:.0f} ms')
print(f'  Двухпоточно  : {t_double*1000:.0f} ms')
print(f'  Ускорение    : {t_single/t_double:.2f}x  (ожидаем ~1.0x из-за GIL)')
print()
print('GIL не даёт параллельного ускорения на CPU-bound Python-коде.')


In [ ]:
%%cython --quiet
# Cython typed + nogil: параллельная игра в кости
# with nogil: отпускает GIL -> потоки работают параллельно
from libc.stdlib cimport rand, srand

def cy_dice_nogil(int n_rounds, int seed=42):
    cdef int i, wins = 0
    cdef int die1, die2
    srand(seed)
    with nogil:          # GIL отпущен — другие потоки могут работать
        for i in range(n_rounds):
            die1 = rand() % 6 + 1
            die2 = rand() % 6 + 1
            if die1 + die2 >= 7:
                wins += 1
    return <double>wins / n_rounds


In [ ]:
import threading, time

N_MC = 2_000_000

# Однопоточно: два вызова подряд
t0 = time.perf_counter()
cy_dice_nogil(N_MC, seed=42)
cy_dice_nogil(N_MC, seed=43)
t_single_cy = time.perf_counter() - t0

# Двухпоточно — с nogil потоки работают параллельно!
results = []
def run_dice(n, seed):
    results.append(cy_dice_nogil(n, seed=seed))

t0 = time.perf_counter()
t1 = threading.Thread(target=run_dice, args=(N_MC, 42))
t2 = threading.Thread(target=run_dice, args=(N_MC, 43))
t1.start(); t2.start()
t1.join();  t2.join()
t_double_cy = time.perf_counter() - t0

print(f'Cython typed + nogil: игра в кости (N={N_MC:,} x2):')
print(f'  Однопоточно  : {t_single_cy*1000:.0f} ms')
print(f'  Двухпоточно  : {t_double_cy*1000:.0f} ms')
print(f'  Ускорение    : {t_single_cy/t_double_cy:.2f}x  (ожидаем ~2.0x)')
print(f'  Результаты   : {[f"{r:.4f}" for r in results]}  (оба ~0.5833)')
print()
print('with nogil: позволяет потокам работать параллельно!')
print('Это то, что Python 3.13 free-threading делает для всего кода.')


In [ ]:
# Визуализация: GIL vs nogil параллелизм
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GIL vs nogil: параллелизм потоков', fontsize=14, fontweight='bold')

# Левый: диаграмма Ганта — GIL
ax1 = axes[0]
ax1.set_title('Python threads + GIL\n(CPU-bound)', fontweight='bold')
ax1.barh(1, 0.5, left=0.0, height=0.4, color='#e74c3c', label='Thread 1 (running)')
ax1.barh(2, 0.5, left=0.5, height=0.4, color='#3498db', label='Thread 2 (running)')
ax1.barh(1, 0.5, left=0.5, height=0.4, color='#e74c3c', alpha=0.2)
ax1.barh(2, 0.5, left=0.0, height=0.4, color='#3498db', alpha=0.2)
ax1.set_yticks([1, 2])
ax1.set_yticklabels(['Thread 1', 'Thread 2'])
ax1.set_xlabel('Время')
ax1.set_xlim(0, 1)
ax1.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax1.text(0.25, 2.6, 'GIL у T1', ha='center', fontsize=9, color='#e74c3c')
ax1.text(0.75, 2.6, 'GIL у T2', ha='center', fontsize=9, color='#3498db')
ax1.text(0.5, 0.3, 'Итого: ~2x медленнее чем надо', ha='center', fontsize=9, color='gray')
ax1.set_ylim(0, 3)
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(axis='x', alpha=0.3)

# Правый: диаграмма Ганта — nogil
ax2 = axes[1]
ax2.set_title('Cython with nogil:\nпараллельное выполнение', fontweight='bold')
ax2.barh(1, 1.0, left=0.0, height=0.4, color='#e74c3c', label='Thread 1 (nogil)')
ax2.barh(2, 1.0, left=0.0, height=0.4, color='#3498db', label='Thread 2 (nogil)')
ax2.set_yticks([1, 2])
ax2.set_yticklabels(['Thread 1', 'Thread 2'])
ax2.set_xlabel('Время')
ax2.set_xlim(0, 1)
ax2.text(0.5, 0.3, 'Итого: ~2x быстрее (настоящий параллелизм)', ha='center', fontsize=9, color='#27ae60')
ax2.set_ylim(0, 3)
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()
print()
print('Ключевой вывод:')
print('  Python threads + GIL  -> последовательно (не параллельно)')
print('  Cython with nogil     -> параллельно (настоящий multi-core)')
print('  Python 3.13 no-GIL    -> параллельно для всего Python-кода')
print()
print('Cython with nogil: = Python 3.13 free-threading, но уже сейчас,')
print('только для блоков с C-типами.')


### Как работает `with nogil:` внутри

```cython
with nogil:
    # Cython генерирует:
    # Py_BEGIN_ALLOW_THREADS   <- сохраняет состояние потока, отпускает GIL
    for i in range(n):
        x = rand() * inv       # чистый C, нет PyObject*
    # Py_END_ALLOW_THREADS     <- захватывает GIL обратно
```

**Правила `nogil` блока:**
- Нельзя создавать/использовать Python-объекты
- Нельзя вызывать Python-функции
- Нельзя обращаться к Python-исключениям
- Можно: C-функции, C-типы, typed memoryviews

### Python 3.13 Free-threading (PEP 703)

```bash
# Установить Python 3.13 без GIL:
# Windows: python.org -> 3.13 -> 'free-threaded'
# Linux:   pyenv install 3.13t

python3.13t -c "import sys; print(sys._is_gil_enabled())"  # False!
```

В Python 3.13t **весь** Python-код может работать параллельно.
Цена: ~10-40% overhead на однопоточный код (атомарные операции вместо GIL).

Cython `with nogil:` — это **та же идея**, но только для C-блоков, без overhead.


---
## 8  Память: list vs NumPy — измерение tracemalloc

Утверждение: Python list хранит каждый float как PyFloatObject (24 байта).
NumPy array хранит как C double (8 байт). Докажем через tracemalloc:


In [ ]:
import tracemalloc, numpy as np

N = 128

tracemalloc.start()
A_ls = [[float(i*N+j) for j in range(N)] for i in range(N)]
_, peak_list = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
A_np = np.arange(N*N, dtype=np.float64).reshape(N, N)
_, peak_np = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'Матрица {N}x{N} = {N*N:,} элементов')
print(f'  Python list of lists : {peak_list/1024:7.1f} KB  ({peak_list/(N*N):.1f} байт/элемент)')
print(f'  NumPy float64        : {peak_np/1024:7.1f} KB  ({peak_np/(N*N):.1f} байт/элемент)')
print(f'  Теоретически: PyFloatObject=24B, double=8B')
print(f'  Экономия памяти: {peak_list/peak_np:.1f}x')


---
## 9  Ограничения Cython

| Ограничение | Подробности |
|---|---|
| **Обязательный build step** | `.pyx` нужно компилировать |
| **Нужен C-компилятор** | MSVC на Windows, gcc/clang на Linux/macOS |
| **Отладка сложна** | Трейсбеки указывают на сгенерированный C |
| **GIL нужен для Python-объектов** | `nogil` только если все переменные — C-типы |
| **Нет JIT** | В отличие от PyPy/Numba |
| **Typed memoryviews требуют contiguous** | Нельзя передать произвольный срез |

### Когда использовать Cython

| Сценарий | Лучший инструмент |
|---|---|
| Тесные числовые циклы | **Cython typed** |
| Обёртка C/C++ библиотеки | **Cython** или **pybind11** |
| Матричная математика | **NumPy** (BLAS, SIMD) |
| JIT без шага сборки | **Numba** `@njit` |

---
## Итог

```
Pure Python  -->  Cython untyped  -->  Cython typed  -->  C
   1x               ~1.3x              ~100x           ~100x
```

- **Cython untyped** — убирает dispatch loop, boxing остаётся → ~1.3x
- **Cython typed** — убирает всё: boxing, GC, GIL → ~100x
- **C** — разница с Cython typed < 5%

**Правило:** профилируй → найди горячий цикл → добавь `cdef` + memoryview → измерь
